<a href="https://colab.research.google.com/github/Hassnkhan0707/-LangGraph_to-build-Chatbot/blob/main/LangGraph_to_build_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
pciutils is already the newest version (1:3.10.0-2build1).
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time

# Start Ollama server in the background
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!ollama pull deepseek-r1

In [ ]:
!ollama list

NAME                  ID              SIZE      MODIFIED       
deepseek-r1:latest    6995872bfe4c    5.2 GB    37 seconds ago    


In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="deepseek-r1")

In [ ]:
import re
from typing import List, Dict, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_ollama.llms import OllamaLLM

class State(TypedDict):
    messages: List[Dict[str, str]]

llm = OllamaLLM(model="deepseek-r1")

def chatbot(state: State):
    response = llm.invoke(state["messages"])
    # Remove deepseek-r1's reasoning block
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    return {"messages": state["messages"] + [{"role": "assistant", "content": response}]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()

def stream_graph_updates(user_input: str):
    state = {"messages": [{"role": "user", "content": user_input}]}
    for event in graph.stream(state):
        for value in event.values():
            print("Assistant:", value["messages"][-1]["content"])

while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    stream_graph_updates(user_input)

/usr/local/lib/python3.13/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


User: what is computer?
Assistant: Okay, let's break down what a computer is.

At its most fundamental level, **a computer is an electronic device designed to store, retrieve, and process data according to a set of instructions (a program) to perform a wide variety of tasks.**

This definition covers both the physical hardware and the concept of computation. Here's a more detailed breakdown:

## 1. The Core Idea

*   **Information Processing:** Computers take in data (information), perform operations on it (processing), and produce useful output. Think of it like a sophisticated calculator or a human brain following instructions.
*   **Programmability:** This is key. Computers can be instructed to perform different tasks by changing the program (software) loaded onto them. They don't just do one thing permanently.

## 2. Key Components (Hardware)

A computer, regardless of type, generally has several essential parts:

*   **Central Processing Unit (CPU):** Often called the "brain" of t